In [1]:
import numpy as np
import pandas as pd
import gc
import os
import json
import nltk
nltk.download('punkt')  # Download the sentence tokenizer
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/zixuanwu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/zixuanwu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
import json
f= open("filtered_metadata.json")
filtered_data = json.load(f)

In [4]:
df = pd.DataFrame(filtered_data)

df["update_date"] = pd.to_datetime(df["update_date"])

In [5]:
import pandas as pd

# Create initial DataFrame
df["doc_id"] = df["id"]

# Step 1: Extract (doc_id, stat_category) pairs
rows = []
for entry in filtered_data:
    doc_id = entry['id']
    abstract = entry['abstract']
    categories = [cat for cat in entry['categories'].split() if cat.startswith("stat.")]
    if len(categories) == 1:  # Keep only if there's exactly one stat category
        rows.append({
            "doc_id": doc_id,
            "abstract": abstract,
            "stat_category": categories[0],
            "update_date": entry["update_date"]
        })

# Step 2: Create DataFrame with unique (abstract, label) pairs
df_unique = pd.DataFrame(rows).drop_duplicates(subset=["doc_id", "stat_category"])


In [6]:
df_unique.shape

(15278, 4)

In [7]:
df_unique = df_unique.loc[df_unique["update_date"] > "2021-01-01"]
df_unique = df_unique.loc[df_unique["stat_category"] != "stat.OT"]

In [8]:
df_sampled = (
    df_unique.groupby("stat_category", group_keys=False)
      .apply(lambda x: x.sample(n=200, replace=False))
      .sample(frac=1, random_state=42)   # optional: shuffle all groups together
)

In [9]:
# Step 3: Output lists
abstracts = df_sampled["abstract"].tolist()
labels = df_sampled["stat_category"].tolist()


In [10]:
df_train = df_sampled.iloc[:800, :].reset_index(drop = True)
df_test = df_sampled.iloc[800:, :].reset_index(drop = True)


In [11]:
import pandas as pd

df_train.to_csv("train.csv", index = False)
df_test.to_csv("test.csv", index = False)